In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
os.chdir('./ci_eval/')
import sys
sys.path.append('python_scripts/')
from subgraph_scripts import plot_subgraph_network, get_subgraph_matrix
# import proplot

In [ ]:
kg_12=['78.875_24.375', '82.125_23.875', '82.875_27.625', '79.125_23.875','88.875_25.375']
kg_1=['-64.625_0.875', '-73.125_-6.875', '-60.125_-2.375', '-60.625_-4.125', '-66.625_-3.125']
kg_2=['-53.625_-3.625', '-64.125_-6.875', '-55.125_1.375', '-59.875_-7.125', '-72.125_-10.125']
kg_4=['140.125_-33.125',  '143.125_-32.375', '145.125_-29.625', '144.375_-33.625', '144.375_-28.125']
kg_7=['148.125_-29.125', '146.625_-31.375', '146.875_-25.875', '147.125_-27.375', '150.875_-28.125']
kg_26=['26.125_46.875', '18.875_48.625', '15.325_47.625', '19.875_43.325', '16.875_49.375']
kg_14=['16.625_45.875',  '17.625_46.625',  '20.325_44.325', '23.125_43.325', '26.325_42.325']
kg_14_2=['-92.875_37.125', '-93.375_32.625', '-98.125_34.625', '-86.625_35.125', '-82.875_37.125']
kg_25=['-92.875_39.875', '-95.625_42.625', '-88.625_40.125', '-92.875_43.825', '-98.325_42.125'];
kg_zone_list=[kg_1, kg_2, kg_4, kg_7, kg_12, kg_14, kg_14_2, kg_25, kg_26]
kg_zone_names=['kg_1', 'kg_2', 'kg_4', 'kg_7', 'kg_12', 'kg_14', 'kg_14_2', 'kg_25', 'kg_26'];
varlingam_results_dir='./ci_eval/plots/dircted_adj_results/varlingam_results/';
pcmci_results_dir='./ci_eval/dircted_adj_results/pcmci_results/';
nse_results_dir='./ci_eval/dircted_adj_results/nse_results/';
pearson_results_dir='./ci_eval/dircted_adj_results/pearson_results/';
dynotears_results_dir='./ci_eval/dircted_adj_results/dynotears_results/';
tcdf_results_dir='./ci_eval/dircted_adj_results/tcdf_results/';
methods=['pearson', 'tcdf', 'varlingam', 'pcmci', 'dynotears',]

In [ ]:
method_zone_wise_results_dict={method:{zone:{} for zone in kg_zone_names} for method in methods}
print(method_zone_wise_results_dict.keys())
print(method_zone_wise_results_dict['dynotears'].keys())
for method in method_zone_wise_results_dict.keys():
    for zone, point_list in zip(method_zone_wise_results_dict[method], kg_zone_list):
        # print(method, zone)
        method_zone_wise_results_dict[method][zone]={grid_point:{} for grid_point in point_list}
print(method_zone_wise_results_dict['dynotears']['kg_1'].keys())

In [ ]:
var_names_nice=pd.read_excel('data/adjacency_matrix/clsm_tru_directional_working.xlsx', sheet_name=1,)['Unnamed: 0'].to_list()
var_names=pd.read_csv('data/adjacency_matrix/benchmark_GLDAS_tru_directed_adj_no_snow_with_lag1.csv', index_col=0).index.to_list()
len(var_names), len(var_names_nice)

In [ ]:
ytick_labels=['Tropical \nrainforest',
    'Tropical \nmonsoon',
    'Arid, \ndesert, hot',
    'Arid, \nsteppe, cold',
    'Temperate, dry winter,\nwarm summer',
    'Temperate, no dry season,\nhot summer',
    'Temperate, no dry season,\nhot summer',
    'Cold, no dry season,\nhot summer',
    'Cold, no dry season,\nwarm summer'];

#### Working

In [ ]:
# using the true adjacency matrix, we will extract the subgraph for target variable 
# select target variable and extract its neighbours
# Qs_tavg SoilMoist_S_tavg Tws_tavg
target_var='SoilMoist_S_tavg'
adj_true_df=pd.read_csv('data/adjacency_matrix/benchmark_GLDAS_tru_directed_adj_no_snow_with_lag1.csv', index_col=0)
df_true = get_subgraph_matrix(target_var, adj_true_df)

In [ ]:
# create a dictionary of drivers, first list drivers from each grid and then remove the redudant drivers
# methods=['pearson', 'tcdf', 'varlingam', 'pcmci', 'dynotears',]
method='pcmci'
union_list = []
for key1 in method_zone_wise_results_dict[method]:
    for key2 in method_zone_wise_results_dict[method][key1]:
        # Read the results of pcmci
        adj_method_df = pd.read_csv(f'./ci_eval/dircted_adj_results/{method}_results/' + key2 + '_abs_adj.csv', index_col=0)
        subgraph_matrix_method = get_subgraph_matrix(target_var, adj_method_df)
        union_list.extend(subgraph_matrix_method[target_var].index.values)
del key1, key2

In [ ]:
# create a dataframe to store the results
temp_df=pd.DataFrame(np.zeros([45,len(sorted(set(union_list+list(df_true.columns))))]), columns=sorted(set(union_list+list(df_true.columns))))
temp_df.head(5)
index=0
for key1 in method_zone_wise_results_dict[method]:
    for key2 in method_zone_wise_results_dict[method][key1]:
        adj_method_df = pd.read_csv(f'./ci_eval/dircted_adj_results/{method}_results/' + key2 + '_abs_adj.csv', index_col=0)
        # get the subgraph matrix for the target variable and its neighbours
        subgraph_matrix_method = get_subgraph_matrix(target_var, adj_method_df)
        # merge it with the temp_df
        temp_df.iloc[index] = temp_df.iloc[index].add(subgraph_matrix_method[target_var], fill_value=0)
        index+=1
print(list(df_true.columns))
print(temp_df.shape)
temp_df.head(5)

In [ ]:
temp_df_full=pd.DataFrame(np.zeros([45, len(var_names)]), columns=var_names);
temp_df_full.head(5)

In [ ]:
merged_df = temp_df.add(temp_df_full, fill_value=0)
sns.heatmap(merged_df, cmap='Blues', cbar=True, xticklabels=1, yticklabels=1)
print(merged_df.shape)
merged_df.head(5)

In [ ]:
ordered_columns = list(dict.fromkeys(list(df_true.index) + list(temp_df_full.columns)))

# # reorder the columns
merged_df = merged_df.loc[:, ordered_columns]
merged_df.head(5)

In [ ]:
# Remove the target variable column from merged_df
if target_var in merged_df.columns:
    merged_df = merged_df.drop(columns=[target_var], axis=1)
merged_df.head(5)

In [ ]:
plt.figure(figsize=(15, 4))
# sns.heatmap(temp_df, cmap='coolwarm', cbar=True,)
sns.heatmap(merged_df, cmap='coolwarm', cbar=True,)
plt.tight_layout()

In [ ]:
var_names_nice=pd.read_excel('data/adjacency_matrix/clsm_tru_directional_working.xlsx', sheet_name=1,)['Unnamed: 1'].to_list()
var_names_nice=var_names_nice+[f'{v}_lag1' for v in var_names_nice]
var_names=pd.read_csv('data/adjacency_matrix/benchmark_GLDAS_tru_directed_adj_no_snow_with_lag1.csv', index_col=0).index.to_list()
len(var_names), len(var_names_nice)

In [ ]:
var_mapping = dict(zip(var_names, var_names_nice))
print(var_mapping)
merged_df.columns = [var_mapping.get(col , col) for col in merged_df.columns]
merged_df.head(5)

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
color_pallette=[[204,204,204],[49,130,189],]
color_pallette = [(round(r/256, 4), round(g/256, 4), round(b/256, 4)) for r, g, b in color_pallette]

fig, axes = plt.subplots(1, 1, figsize=(15, 8), dpi=300)
sns.heatmap(merged_df,
            cmap=color_pallette,
            cbar=False, ax=axes, linewidths=0.5, linecolor='white', 
            square=True,
             xticklabels=True, yticklabels=False, )
plt.tick_params(labelsize=7, labelbottom = False, bottom=False, top = False, labeltop=True, rotation=90)

# make all column names with '_lag1' to be italics
for label in axes.get_xticklabels():
    if '_lag1' in label.get_text():
        label.set_fontstyle('italic')
    else:
        pass
axes.xaxis.set_minor_locator(plt.NullLocator())
axes.set_yticks(np.arange(len(ytick_labels)) * 5 + 2.5, )
axes.set_yticklabels(ytick_labels, rotation=0, )
axes.yaxis.set_minor_locator(plt.NullLocator())
for i in range(5, len(merged_df), 5):
    axes.axhline(i, color='black', lw=1)
# axes.plot([int(len(merged_df.columns)-len(remaining_columns)), int(len(merged_df.columns)-len(remaining_columns))], [0, 45], color='#b30000', lw=1)
axes.plot([int(len(df_true.index)), int(len(df_true.index))], [0, 45], color='#b30000', lw=1)
axes.set_title(f'{method}', fontsize=9, style='italic')
# plt.savefig(f'./ci_eval/plots/figs/small_picture_figs/pearson_{target_var}.png', bbox_inches='tight', dpi=300)
plt.tight_layout()
plt.show()

### Production
### Now a for loop for all the 5 cd methods

In [ ]:
# using the true adjacency matrix, we will extract the subgraph for target variable 
# select target variable and extract its neighbours
# Qs_tavg SoilMoist_S_tavg Tws_tavg
target_var='SoilMoist_S_tavg'
adj_true_df=pd.read_csv('data/adjacency_matrix/benchmark_GLDAS_tru_directed_adj_no_snow_with_lag1.csv', index_col=0)
df_true = get_subgraph_matrix(target_var, adj_true_df)

In [ ]:
var_names_nice=pd.read_excel('data/adjacency_matrix/clsm_tru_directional_working.xlsx', sheet_name=1,)['Unnamed: 1'].to_list()
var_names_nice=var_names_nice+[f'{v}_lag1' for v in var_names_nice]
var_names=pd.read_csv('data/adjacency_matrix/benchmark_GLDAS_tru_directed_adj_no_snow_with_lag1.csv', index_col=0).index.to_list()
var_mapping = dict(zip(var_names, var_names_nice))
print(var_mapping, '\n', len(var_names), len(var_names_nice))

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
color_pallette=[[204,204,204],[49,130,189],]

color_pallette = [(round(r/256, 4), round(g/256, 4), round(b/256, 4)) for r, g, b in color_pallette]
method_names=['PCC', 'TCDF', 'VARLiNGAM', 'PCMCI+', 'DYNOTEARS',]
for index2, method in enumerate(method_zone_wise_results_dict.keys()):
    # if method == 'pearson':
    #     continue
    #1
    # create a dictionary of drivers, first list drivers from each grid and then remove the redudant drivers
    union_list = []
    for key1 in method_zone_wise_results_dict[method]:
        for key2 in method_zone_wise_results_dict[method][key1]:
            # Read the results of pcmci
            adj_method_df = pd.read_csv(f'./ci_eval/dircted_adj_results/{method}_results/' + key2 + '_abs_adj.csv', index_col=0)
            subgraph_matrix_method = get_subgraph_matrix(target_var, adj_method_df)
            union_list.extend(subgraph_matrix_method[target_var].index.values)
    #2
    # create a dataframe to store the results
    temp_df=pd.DataFrame(np.zeros([45,len(sorted(set(union_list+list(df_true.columns))))]), columns=sorted(set(union_list+list(df_true.columns))))
    temp_df.head(5)
    index=0
    for key1 in method_zone_wise_results_dict[method]:
        for key2 in method_zone_wise_results_dict[method][key1]:
            adj_method_df = pd.read_csv(f'./ci_eval/dircted_adj_results/{method}_results/' + key2 + '_abs_adj.csv', index_col=0)
            # get the subgraph matrix for the target variable and its neighbours
            subgraph_matrix_method = get_subgraph_matrix(target_var, adj_method_df)
            # merge it with the temp_df
            temp_df.iloc[index] = temp_df.iloc[index].add(subgraph_matrix_method[target_var], fill_value=0)
            index+=1
    # print(list(df_true.columns))
    
    # 2.5 make a new dataframe with all the variables (27*2)

    temp_df_full=pd.DataFrame(np.zeros([45, len(var_names)]), columns=var_names);
    # merge it with the temp_df
    merged_df = temp_df.add(temp_df_full, fill_value=0)

    #3
    # Reorder the columns to put the true drivers first
    ordered_columns = list(dict.fromkeys(list(df_true.index) + list(temp_df_full.columns)))
    # # reorder the columns
    merged_df = merged_df.loc[:, ordered_columns]

    # Remove the target variable column from merged_df
    if target_var in merged_df.columns:
        merged_df = merged_df.drop(columns=[target_var], axis=1)

    merged_df.to_csv(f'./ci_eval/plots/figs/small_picture_figs/{method}_{target_var}_nomapping.csv', index=False)
    # Replace the column names to their full definitions
    merged_df.columns = [var_mapping.get(col , col) for col in merged_df.columns]
    
    #4
    fig, axes = plt.subplots(1, 1, figsize=(8*6/5, 8))
    sns.heatmap(merged_df,
                cmap=color_pallette,
                cbar=False, ax=axes, linewidths=0.5, linecolor='white', square=True,
                xticklabels=True, yticklabels=False, )
    # if method == 'pearson':
    #     plt.tick_params(labelsize=12, labelbottom = False, bottom=False, top = False, labeltop=True, rotation=90)
    #     # make all column names with '_lag1' to be italics
    #     for label in axes.get_xticklabels():
    #         if '_lag1' in label.get_text():
    #             label.set_fontstyle('italic')
    #         else:
    #             pass
    #     axes.xaxis.set_minor_locator(plt.NullLocator())
    # else:
    #     plt.tick_params(labelsize=12, labelbottom = False, bottom=False, top = False, labeltop=False, rotation=90)
    plt.tick_params(labelsize=12, labelbottom = False, bottom=False, top = False, labeltop=False, rotation=90)

    axes.set_yticks(np.arange(len(ytick_labels)) * 5 + 2.5, )
    axes.set_yticklabels(ytick_labels, rotation=0, )
    axes.yaxis.set_minor_locator(plt.NullLocator())
    for i in range(5, len(merged_df), 5):
        axes.axhline(i, color='black', lw=1, ls='-')
    axes.plot([int(len(df_true.index)), int(len(df_true.index))], [0, 45], color='#b30000', lw=1)
    # axes.set_title(f'{method_names[index2]}', fontsize=12, )
    axes.text(1.01, 0.5, f'{method_names[index2]}', transform=axes.transAxes, fontsize=16, rotation=270, verticalalignment='center',
            horizontalalignment='center', fontweight='bold')
    plt.savefig(f'./ci_eval/plots/figs/small_picture_figs/{method}_{target_var}.svg', 
                        bbox_inches='tight', dpi=400, pad_inches=0)
    merged_df.to_csv(f'./ci_eval/plots/figs/small_picture_figs/{method}_{target_var}.csv', index=False)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
color_pallette=[[204,204,204],[49,130,189],]

color_pallette = [(round(r/256, 4), round(g/256, 4), round(b/256, 4)) for r, g, b in color_pallette]
method_names=['PCC', 'TCDF', 'VARLiNGAM', 'PCMCI+', 'DYNOTEARS',]
index2, method = 0, 'pearson'
#1
# create a dictionary of drivers, first list drivers from each grid and then remove the redudant drivers
union_list = []
for key1 in method_zone_wise_results_dict[method]:
    for key2 in method_zone_wise_results_dict[method][key1]:
        # Read the results of pcmci
        adj_method_df = pd.read_csv(f'./ci_eval/dircted_adj_results/{method}_results/' + key2 + '_abs_adj.csv', index_col=0)
        subgraph_matrix_method = get_subgraph_matrix(target_var, adj_method_df)
        union_list.extend(subgraph_matrix_method[target_var].index.values)
#2
# create a dataframe to store the results
temp_df=pd.DataFrame(np.zeros([45,len(sorted(set(union_list+list(df_true.columns))))]), columns=sorted(set(union_list+list(df_true.columns))))
temp_df.head(5)
index=0
for key1 in method_zone_wise_results_dict[method]:
    for key2 in method_zone_wise_results_dict[method][key1]:
        adj_method_df = pd.read_csv(f'./ci_eval/dircted_adj_results/{method}_results/' + key2 + '_abs_adj.csv', index_col=0)
        # get the subgraph matrix for the target variable and its neighbours
        subgraph_matrix_method = get_subgraph_matrix(target_var, adj_method_df)
        # merge it with the temp_df
        temp_df.iloc[index] = temp_df.iloc[index].add(subgraph_matrix_method[target_var], fill_value=0)
        index+=1
# print(list(df_true.columns))

# 2.5 make a new dataframe with all the variables (27*2)

temp_df_full=pd.DataFrame(np.zeros([45, len(var_names)]), columns=var_names);
# merge it with the temp_df
merged_df = temp_df.add(temp_df_full, fill_value=0)

#3
# Reorder the columns to put the true drivers first
ordered_columns = list(dict.fromkeys(list(df_true.index) + list(temp_df_full.columns)))
# # reorder the columns
merged_df = merged_df.loc[:, ordered_columns]

# Remove the target variable column from merged_df
if target_var in merged_df.columns:
    merged_df = merged_df.drop(columns=[target_var], axis=1)

# Replace the column names to their full definitions
merged_df.columns = [var_mapping.get(col , col) for col in merged_df.columns]

#4
fig, axes = plt.subplots(1, 1, figsize=(8*6/5, 12))
sns.heatmap(merged_df,
            cmap=color_pallette,
            cbar=False, ax=axes, linewidths=0.5, linecolor='white', square=True,
            xticklabels=True, yticklabels=False, )
plt.tick_params(labelsize=12, labelbottom = False, bottom=False, top = False, labeltop=True, rotation=90)
# make all column names with '_lag1' to be italics
for label in axes.get_xticklabels():
    if '_lag1' in label.get_text():
        label.set_fontstyle('italic')
    else:
        pass
axes.xaxis.set_minor_locator(plt.NullLocator())

axes.set_yticks(np.arange(len(ytick_labels)) * 5 + 2.5, )
axes.set_yticklabels(ytick_labels, rotation=0, )
axes.yaxis.set_minor_locator(plt.NullLocator())
for i in range(5, len(merged_df), 5):
    axes.axhline(i, color='black', lw=1, ls='-')
axes.plot([int(len(df_true.index)), int(len(df_true.index))], [0, 45], color='#b30000', lw=1)

axes.text(1.01, 0.5, f'{method_names[index2]}', transform=axes.transAxes, fontsize=16, rotation=270, verticalalignment='center',
        horizontalalignment='center', fontweight='bold')
plt.savefig(f'./ci_eval/plots/figs/small_picture_figs/{method}_{target_var}.svg', 
                    bbox_inches='tight', dpi=400, pad_inches=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
color_pallette=[[204,204,204],[49,130,189],]

color_pallette = [(round(r/256, 4), round(g/256, 4), round(b/256, 4)) for r, g, b in color_pallette]
method_names=['PCC', 'TCDF', 'VARLiNGAM', 'PCMCI+', 'DYNOTEARS',]
index2, method = 1, 'tcdf'
#1
# create a dictionary of drivers, first list drivers from each grid and then remove the redudant drivers
union_list = []
for key1 in method_zone_wise_results_dict[method]:
    for key2 in method_zone_wise_results_dict[method][key1]:
        # Read the results of pcmci
        adj_method_df = pd.read_csv(f'./ci_eval/dircted_adj_results/{method}_results/' + key2 + '_abs_adj.csv', index_col=0)
        subgraph_matrix_method = get_subgraph_matrix(target_var, adj_method_df)
        union_list.extend(subgraph_matrix_method[target_var].index.values)
#2
# create a dataframe to store the results
temp_df=pd.DataFrame(np.zeros([45,len(sorted(set(union_list+list(df_true.columns))))]), columns=sorted(set(union_list+list(df_true.columns))))
temp_df.head(5)
index=0
for key1 in method_zone_wise_results_dict[method]:
    for key2 in method_zone_wise_results_dict[method][key1]:
        adj_method_df = pd.read_csv(f'./ci_eval/dircted_adj_results/{method}_results/' + key2 + '_abs_adj.csv', index_col=0)
        # get the subgraph matrix for the target variable and its neighbours
        subgraph_matrix_method = get_subgraph_matrix(target_var, adj_method_df)
        # merge it with the temp_df
        temp_df.iloc[index] = temp_df.iloc[index].add(subgraph_matrix_method[target_var], fill_value=0)
        index+=1
# print(list(df_true.columns))

# 2.5 make a new dataframe with all the variables (27*2)
temp_df_full=pd.DataFrame(np.zeros([45, len(var_names)]), columns=var_names);
# merge it with the temp_df
merged_df = temp_df.add(temp_df_full, fill_value=0)

#3
# Reorder the columns to put the true drivers first
ordered_columns = list(dict.fromkeys(list(df_true.index) + list(temp_df_full.columns)))
# # reorder the columns
merged_df = merged_df.loc[:, ordered_columns]

# Remove the target variable column from merged_df
if target_var in merged_df.columns:
    merged_df = merged_df.drop(columns=[target_var], axis=1)

# Replace the column names to their full definitions
merged_df.columns = [var_mapping.get(col , col) for col in merged_df.columns]

#4
fig, axes = plt.subplots(1, 1, figsize=(8*6/5, 12))
sns.heatmap(merged_df,
            cmap=color_pallette,
            cbar=False, ax=axes, linewidths=0.5, linecolor='white', square=True,
            xticklabels=True, yticklabels=False, )
plt.tick_params(labelsize=12, labelbottom = False, bottom=False, top = False, labeltop=True, rotation=90)
# make all column names with '_lag1' to be italics
for label in axes.get_xticklabels():
    if '_lag1' in label.get_text():
        label.set_fontstyle('italic')
    else:
        pass
axes.xaxis.set_minor_locator(plt.NullLocator())

# axes.set_yticks(np.arange(len(ytick_labels)) * 5 + 2.5, )
# axes.set_yticklabels(ytick_labels, rotation=0, )
axes.set_yticklabels([])
axes.yaxis.set_minor_locator(plt.NullLocator())
for i in range(5, len(merged_df), 5):
    axes.axhline(i, color='black', lw=1, ls='-')
axes.plot([int(len(df_true.index)), int(len(df_true.index))], [0, 45], color='#b30000', lw=1)

axes.text(1.01, 0.5, f'{method_names[index2]}', transform=axes.transAxes, fontsize=16, rotation=270, verticalalignment='center',
        horizontalalignment='center', fontweight='bold')
# plt.savefig(f'./ci_eval/plots/figs/small_picture_figs/{method}_{target_var}.svg', 
                    # bbox_inches='tight', dpi=400, pad_inches=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
color_pallette=[[204,204,204],[49,130,189],]

color_pallette = [(round(r/256, 4), round(g/256, 4), round(b/256, 4)) for r, g, b in color_pallette]
method_names=['PCC', 'TCDF', 'VARLiNGAM', 'PCMCI+', 'DYNOTEARS',]
index2, method = 3, 'pcmci'
#1
# create a dictionary of drivers, first list drivers from each grid and then remove the redudant drivers
union_list = []
for key1 in method_zone_wise_results_dict[method]:
    for key2 in method_zone_wise_results_dict[method][key1]:
        # Read the results of pcmci
        adj_method_df = pd.read_csv(f'./ci_eval/dircted_adj_results/{method}_results/' + key2 + '_abs_adj.csv', index_col=0)
        subgraph_matrix_method = get_subgraph_matrix(target_var, adj_method_df)
        union_list.extend(subgraph_matrix_method[target_var].index.values)
#2
# create a dataframe to store the results
temp_df=pd.DataFrame(np.zeros([45,len(sorted(set(union_list+list(df_true.columns))))]), columns=sorted(set(union_list+list(df_true.columns))))
temp_df.head(5)
index=0
for key1 in method_zone_wise_results_dict[method]:
    for key2 in method_zone_wise_results_dict[method][key1]:
        adj_method_df = pd.read_csv(f'./ci_eval/dircted_adj_results/{method}_results/' + key2 + '_abs_adj.csv', index_col=0)
        # get the subgraph matrix for the target variable and its neighbours
        subgraph_matrix_method = get_subgraph_matrix(target_var, adj_method_df)
        # merge it with the temp_df
        temp_df.iloc[index] = temp_df.iloc[index].add(subgraph_matrix_method[target_var], fill_value=0)
        index+=1
# print(list(df_true.columns))

# 2.5 make a new dataframe with all the variables (27*2)

temp_df_full=pd.DataFrame(np.zeros([45, len(var_names)]), columns=var_names);
# merge it with the temp_df
merged_df = temp_df.add(temp_df_full, fill_value=0)

#3
# Reorder the columns to put the true drivers first
ordered_columns = list(dict.fromkeys(list(df_true.index) + list(temp_df_full.columns)))
# # reorder the columns
merged_df = merged_df.loc[:, ordered_columns]

# Remove the target variable column from merged_df
if target_var in merged_df.columns:
    merged_df = merged_df.drop(columns=[target_var], axis=1)

# Replace the column names to their full definitions
merged_df.columns = [var_mapping.get(col , col) for col in merged_df.columns]

#4
fig, axes = plt.subplots(1, 1, figsize=(8*6/5, 12))
sns.heatmap(merged_df,
            cmap=color_pallette,
            cbar=False, ax=axes, linewidths=0.5, linecolor='white', square=True,
            xticklabels=True, yticklabels=False, )
plt.tick_params(labelsize=12, labelbottom = False, bottom=False, top = False, labeltop=False, rotation=90)
# make all column names with '_lag1' to be italics
for label in axes.get_xticklabels():
    if '_lag1' in label.get_text():
        label.set_fontstyle('italic')
    else:
        pass
axes.xaxis.set_minor_locator(plt.NullLocator())

# axes.set_yticks(np.arange(len(ytick_labels)) * 5 + 2.5, )
# axes.set_yticklabels(ytick_labels, rotation=0, )
axes.set_yticklabels([])
axes.yaxis.set_minor_locator(plt.NullLocator())
for i in range(5, len(merged_df), 5):
    axes.axhline(i, color='black', lw=1, ls='-')
axes.plot([int(len(df_true.index)), int(len(df_true.index))], [0, 45], color='#b30000', lw=1)

axes.text(1.01, 0.5, f'{method_names[index2]}', transform=axes.transAxes, fontsize=16, rotation=270, verticalalignment='center',
        horizontalalignment='center', fontweight='bold')
# plt.savefig(f'./ci_eval/plots/figs/small_picture_figs/{method}_{target_var}.svg', 
                    # bbox_inches='tight', dpi=400, pad_inches=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
color_pallette=[[204,204,204],[49,130,189],]

color_pallette = [(round(r/256, 4), round(g/256, 4), round(b/256, 4)) for r, g, b in color_pallette]
method_names=['PCC', 'TCDF', 'VARLiNGAM', 'PCMCI+', 'DYNOTEARS',]
# methods=['pearson', 'tcdf', 'varlingam', 'pcmci', 'dynotears',]
index2, method = 4, 'dynotears'
#1
# create a dictionary of drivers, first list drivers from each grid and then remove the redudant drivers
union_list = []
for key1 in method_zone_wise_results_dict[method]:
    for key2 in method_zone_wise_results_dict[method][key1]:
        # Read the results of pcmci
        adj_method_df = pd.read_csv(f'./ci_eval/dircted_adj_results/{method}_results/' + key2 + '_abs_adj.csv', index_col=0)
        subgraph_matrix_method = get_subgraph_matrix(target_var, adj_method_df)
        union_list.extend(subgraph_matrix_method[target_var].index.values)
#2
# create a dataframe to store the results
temp_df=pd.DataFrame(np.zeros([45,len(sorted(set(union_list+list(df_true.columns))))]), columns=sorted(set(union_list+list(df_true.columns))))
temp_df.head(5)
index=0
for key1 in method_zone_wise_results_dict[method]:
    for key2 in method_zone_wise_results_dict[method][key1]:
        adj_method_df = pd.read_csv(f'./ci_eval/dircted_adj_results/{method}_results/' + key2 + '_abs_adj.csv', index_col=0)
        # get the subgraph matrix for the target variable and its neighbours
        subgraph_matrix_method = get_subgraph_matrix(target_var, adj_method_df)
        # merge it with the temp_df
        temp_df.iloc[index] = temp_df.iloc[index].add(subgraph_matrix_method[target_var], fill_value=0)
        index+=1
# print(list(df_true.columns))

# 2.5 make a new dataframe with all the variables (27*2)

temp_df_full=pd.DataFrame(np.zeros([45, len(var_names)]), columns=var_names);
# merge it with the temp_df
merged_df = temp_df.add(temp_df_full, fill_value=0)

#3
# Reorder the columns to put the true drivers first
ordered_columns = list(dict.fromkeys(list(df_true.index) + list(temp_df_full.columns)))
# # reorder the columns
merged_df = merged_df.loc[:, ordered_columns]

# Remove the target variable column from merged_df
if target_var in merged_df.columns:
    merged_df = merged_df.drop(columns=[target_var], axis=1)

# Replace the column names to their full definitions
merged_df.columns = [var_mapping.get(col , col) for col in merged_df.columns]

#4
fig, axes = plt.subplots(1, 1, figsize=(8*6/5, 12))
sns.heatmap(merged_df,
            cmap=color_pallette,
            cbar=False, ax=axes, linewidths=0.5, linecolor='white', square=True,
            xticklabels=True, yticklabels=False, )
plt.tick_params(labelsize=12, labelbottom = False, bottom=False, top = False, labeltop=True, rotation=90)
# make all column names with '_lag1' to be italics
for label in axes.get_xticklabels():
    if '_lag1' in label.get_text():
        label.set_fontstyle('italic')
    else:
        pass
axes.xaxis.set_minor_locator(plt.NullLocator())

axes.set_yticks(np.arange(len(ytick_labels)) * 5 + 2.5, )
axes.set_yticklabels(ytick_labels, rotation=0, )
axes.yaxis.set_minor_locator(plt.NullLocator())
for i in range(5, len(merged_df), 5):
    axes.axhline(i, color='black', lw=1, ls='-')
axes.plot([int(len(df_true.index)), int(len(df_true.index))], [0, 45], color='#b30000', lw=1)

axes.text(1.01, 0.5, f'{method_names[index2]}', transform=axes.transAxes, fontsize=16, rotation=270, verticalalignment='center',
        horizontalalignment='center', fontweight='bold')
# plt.savefig(f'./ci_eval/plots/figs/small_picture_figs/{method}_{target_var}.svg', 
                    # bbox_inches='tight', dpi=400, pad_inches=0)
plt.tight_layout()
plt.show()

##### Small table summarizing the above figures

In [ ]:
merged_dict={f'{method}': [] for method in method_zone_wise_results_dict.keys()}

In [ ]:
for method in merged_dict.keys():
    merged_dict[method] = pd.read_csv(f'./ci_eval/plots/figs/small_picture_figs/{method}_{target_var}.csv')

In [ ]:
merged_dict['tcdf'].head(5)

In [ ]:
summary_table=np.zeros([5,len(df_true.index)+1])
summary_table.shape

In [ ]:
for index, (method, merged_dataframe) in enumerate(merged_dict.items()):
    # print(index, method)
    for col in range(len(df_true.index)):        
        summary_table[index, col] = merged_dataframe.iloc[:, col].sum()/45
    summary_table[index, col+1] = merged_dataframe.iloc[:, col+1:].values.sum()/(45*(53-6))

In [ ]:
# for index, (method, merged_dataframe) in enumerate(merged_dict.items()):
#     # print(index, method)
#     for col in range(len(df_true.index)):
#         summary_table[index, 0] = merged_dataframe.iloc[:, 0].sum()/45
#         summary_table[index, 1] = merged_dataframe.iloc[:, 1].sum()/45
#         summary_table[index, 2] = merged_dataframe.iloc[:, 2].sum()/45
#         summary_table[index, 3] = merged_dataframe.iloc[:, 3].sum()/45
#         summary_table[index, 4] = merged_dataframe.iloc[:, 4].sum()/45
#         summary_table[index, 5] = merged_dataframe.iloc[:, 5].sum()/45
#         summary_table[index, 6] = merged_dataframe.iloc[:, 6:].values.sum()/(45*(53-6))

In [ ]:
merged_dict['tcdf'].columns[0:len(df_true.index)]

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'

# Create figure with appropriate size
fig, ax = plt.subplots(figsize=(4, 6))

# Create a separate legend
norm = plt.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap='Blues', norm=norm)
sm.set_array([])

heatmap_=sns.heatmap(summary_table[:,:-1], 
            xticklabels=merged_dict['tcdf'].columns[0:len(df_true.index)],
            yticklabels=['PCC', 'TCDF', 'VARLiNGAM', 'PCMCI+', 'DYNOTEARS'],
            cbar=False, cmap='Blues', norm=norm,
            annot=True, fmt='.2f', linewidths=0.5, linecolor='white', square=True,
            annot_kws={"size": 12, "weight": "normal", "color": "black"})

for text in heatmap_.texts:
    if float(text.get_text()) < 0.5:
        text.set_color("black")
    else:
        text.set_color("white")

# Add the legend to the figure
cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', pad=0.05, ticks=[0.0, 0.5, 1],)
cbar.set_label('Fraction of grids')

plt.tick_params(labelsize=12, labelbottom=False, bottom=False, top=False, labeltop=True, rotation=90)

# Make all column names with '_lag1' italic and also make the last label italic
for i, label in enumerate(ax.get_xticklabels()):
    if '_lag1' in label.get_text() or i == len(ax.get_xticklabels())-1:
        label.set_fontstyle('italic')

plt.yticks(fontsize=12, rotation=0)
plt.savefig(f'./ci_eval/plots/figs/small_picture_figs/summary_table_truepositives.svg', dpi=400, bbox_inches='tight', pad_inches=0)
# plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'

# Create figure with appropriate size
fig, ax = plt.subplots(figsize=(1, 3))

# Create a separate legend
norm = plt.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap='Reds', norm=norm)
sm.set_array([])

heatmap_=sns.heatmap(summary_table[:,-1:], 
            xticklabels=['False Positives'], 
            yticklabels=[],
            cbar=False, cmap='Reds', norm=norm,
            annot=True, fmt='.2f', linewidths=0.5, linecolor='white', square=True,
            annot_kws={"size": 12, "weight": "normal", "color": "black"})

for text in heatmap_.texts:
    if float(text.get_text()) < 0.5:
        text.set_color("black")
    else:
        text.set_color("white")

# Add the legend to the figure
cbar = fig.colorbar(sm, ax=ax, orientation='vertical', pad=0.1, ticks=[0.0, 0.5, 1],)
cbar.set_label('Fraction of grids')

plt.tick_params(labelsize=12, labelbottom=False, bottom=False, top=False, labeltop=True, rotation=90)

plt.yticks(fontsize=12, rotation=0)
plt.savefig(f'./ci_eval/plots/figs/small_picture_figs/summary_table_falsepositives.svg', dpi=400, bbox_inches='tight', pad_inches=0)
# plt.tight_layout()
plt.show()